In [5]:
# Run this once in your notebook if not already installed
!pip install -q pandas openpyxl ipywidgets

import os
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
def load_screener_data(file_path="companies.xlsx"):
    """Reads and sanitizes numeric columns from the Screener export."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"'{file_path}' not found. Please place it in this notebook's directory.")
    
    df = pd.read_excel(file_path)
    df.columns = df.columns.astype(str).str.strip()
    
    numeric_cols = [
        'CMP Rs.', 'Mar Cap Rs.Cr.', 'P/E', 'Ind PE', 'PEG', 'ROCE %',
        'ROE 5Yr %', 'Debt / Eq', 'Int Coverage', 'CMP / FCF', 'OPM %',
        'Sales Var 3Yrs %', 'Profit Var 5Yrs %', 'Prom. Hold. %', 'Pledged %'
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace(',', '').str.strip()
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
            
    return df


def evaluate_company(row, strategy):
    """Calculates quantitative scores, catalysts, and risk factors."""
    company = str(row.get('Company', 'Unknown'))
    pe = float(row.get('P/E', 0.0))
    ind_pe = float(row.get('Ind PE', 0.0))
    peg = float(row.get('PEG', 0.0))
    roce = float(row.get('ROCE %', 0.0))
    roe_5yr = float(row.get('ROE 5Yr %', 0.0))
    debt_eq = float(row.get('Debt / Eq', 0.0))
    int_cov = float(row.get('Int Coverage', 0.0))
    cmp_fcf = float(row.get('CMP / FCF', 0.0))
    opm = float(row.get('OPM %', 0.0))
    sales_3yr = float(row.get('Sales Var 3Yrs %', 0.0))
    profit_5yr = float(row.get('Profit Var 5Yrs %', 0.0))
    prom_hold = float(row.get('Prom. Hold. %', 0.0))
    pledged = float(row.get('Pledged %', 0.0))

    is_bank = any(term in company.upper() for term in ["BANK", "SBI", "FINANCE"])
    score = 0.0
    catalysts, risks = [], []

    # Governance Check
    if pledged == 0:
        score += 1.0
        catalysts.append("Governance hygiene: Zero promoter shares pledged.")
    else:
        score -= 3.0
        risks.append(f"Encumbrance risk: {pledged}% promoter shares are pledged.")

    # Strategy Evaluation
    if strategy == "Growth at Reasonable Price (GARP)":
        if 0 < peg <= 1.0:
            score += 4.0
            catalysts.append(f"Attractive valuation: PEG ratio of {peg:.2f} (< 1.0).")
        elif peg > 2.0:
            score -= 2.0
            risks.append(f"Valuation stretch: Elevated PEG ratio of {peg:.2f}.")
        if profit_5yr >= 18.0 and sales_3yr >= 10.0:
            score += 3.0
            catalysts.append(f"Consistent compounding: 5Y PAT CAGR {profit_5yr:.1f}% & 3Y Sales {sales_3yr:.1f}%.")
        if pe < ind_pe:
            score += 2.0
            catalysts.append(f"Multiple discount: P/E ({pe:.1f}x) is below industry median ({ind_pe:.1f}x).")

    elif strategy == "High Quality Compounders (Buffett Style)":
        if not is_bank:
            if roce >= 20.0:
                score += 4.0
                catalysts.append(f"Superior capital productivity: ROCE of {roce:.1f}%.")
            if debt_eq <= 0.3:
                score += 3.0
                catalysts.append(f"Conservative balance sheet: Debt/Equity of {debt_eq:.2f}.")
            elif debt_eq > 1.0:
                score -= 2.0
                risks.append(f"High balance sheet leverage: Debt/Equity of {debt_eq:.2f}.")
        if roe_5yr >= 18.0:
            score += 3.0
            catalysts.append(f"Proven economic moat: 5-Year average ROE of {roe_5yr:.1f}%.")
        if opm >= 25.0:
            score += 2.0
            catalysts.append(f"Healthy operating margin: OPM of {opm:.1f}%.")

    elif strategy == "Deep Value & Cash Cows":
        if pe > 0 and ind_pe > 0 and pe <= ind_pe * 0.85:
            score += 4.0
            catalysts.append(f"Substantial valuation margin: P/E {pe:.1f}x vs Industry {ind_pe:.1f}x.")
        if 0 < cmp_fcf <= 20.0:
            score += 4.0
            catalysts.append(f"Strong cash generation: Attractive Price/FCF of {cmp_fcf:.1f}x.")
        elif cmp_fcf > 35.0:
            score -= 2.0
            risks.append(f"Weak cash yield: Price/FCF is high at {cmp_fcf:.1f}x.")

    elif strategy == "Debt-Free / Low-Risk Safe Havens":
        if not is_bank:
            if debt_eq <= 0.2:
                score += 4.0
                catalysts.append(f"Virtually debt-free: Low D/E ratio of {debt_eq:.2f}.")
            if int_cov >= 15.0:
                score += 3.0
                catalysts.append(f"Strong interest coverage buffer: {int_cov:.1f}x.")
        if prom_hold >= 55.0:
            score += 2.0
            catalysts.append(f"High promoter stake: {prom_hold:.1f}%.")

    # Verdict
    if score >= 8.0:
        verdict = "STRONG ACCUMULATE"
    elif score >= 5.0:
        verdict = "ACCUMULATE / BUY"
    elif score >= 2.0:
        verdict = "HOLD / MONITOR"
    else:
        verdict = "UNDERPERFORM / AVOID"

    return {
        "Company": company,
        "CMP Rs.": row.get('CMP Rs.', 0.0),
        "P/E": pe,
        "Ind PE": ind_pe,
        "PEG": peg,
        "ROCE %": roce,
        "ROE 5Yr %": roe_5yr,
        "Debt / Eq": debt_eq,
        "CMP / FCF": cmp_fcf,
        "Score": round(score, 1),
        "Verdict": verdict,
        "Catalysts": catalysts,
        "Risks": risks
    }

In [7]:
def create_printable_report(results, strategy_name, filename="stock_report.html"):
    """Generates an HTML report styled for instant PDF printing."""
    rows_html = ""
    for idx, r in enumerate(results, start=1):
        color = "#059669" if "ACCUMULATE" in r['Verdict'] or "BUY" in r['Verdict'] else "#D97706" if "HOLD" in r['Verdict'] else "#DC2626"
        rows_html += f"""
        <tr>
            <td style="text-align:center;">{idx}</td>
            <td><b>{r['Company']}</b></td>
            <td style="text-align:right;">₹{r['CMP Rs.']:,.2f}</td>
            <td style="text-align:right;">{r['P/E']:.1f}</td>
            <td style="text-align:right;">{r['Ind PE']:.1f}</td>
            <td style="text-align:right;">{r['PEG']:.2f}</td>
            <td style="text-align:right;">{r['ROCE %']:.1f}%</td>
            <td style="text-align:right;">{r['ROE 5Yr %']:.1f}%</td>
            <td style="text-align:center;"><span style="background:{color}; color:white; padding:3px 8px; border-radius:4px; font-weight:bold; font-size:11px;">{r['Verdict']}</span></td>
        </tr>
        """

    details_html = ""
    for r in results:
        cat_items = "".join([f"<li style='color:#065F46;'>{c}</li>" for c in r['Catalysts']]) or "<li>None</li>"
        risk_items = "".join([f"<li style='color:#991B1B;'>{item}</li>" for item in r['Risks']]) or "<li>None</li>"
        details_html += f"""
        <div style="border: 1px solid #E2E8F0; border-radius: 6px; padding: 12px; margin-bottom: 12px; page-break-inside: avoid;">
            <div style="font-size: 14px; font-weight: bold; color: #1E3A8A;">{r['Company']} — <i>{r['Verdict']}</i> (Score: {r['Score']} pts)</div>
            <div style="font-size: 11px; color: #64748B; margin-bottom: 6px;">CMP: ₹{r['CMP Rs.']:,.2f} | P/E: {r['P/E']:.1f}x | Ind P/E: {r['Ind PE']:.1f}x | ROCE: {r['ROCE %']:.1f}% | 5Y ROE: {r['ROE 5Yr %']:.1f}% | D/E: {r['Debt / Eq']:.2f}</div>
            <div style="font-size: 11px; font-weight: bold; color: #065F46;">Growth Catalysts & Moats:</div>
            <ul style="margin: 2px 0 6px 18px; font-size: 11px;">{cat_items}</ul>
            <div style="font-size: 11px; font-weight: bold; color: #991B1B;">Risk Factors & Headwinds:</div>
            <ul style="margin: 2px 0 6px 18px; font-size: 11px;">{risk_items}</ul>
        </div>
        """

    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <title>Stock Report - {strategy_name}</title>
        <style>
            body {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; padding: 24px; color: #1E293B; }}
            h1 {{ color: #1E3A8A; margin-bottom: 2px; font-size: 22px; }}
            .subtitle {{ color: #64748B; font-size: 12px; margin-bottom: 16px; }}
            table {{ width: 100%; border-collapse: collapse; margin-bottom: 20px; font-size: 12px; }}
            th {{ background-color: #1E3A8A; color: white; padding: 8px 6px; text-align: left; }}
            td {{ padding: 7px 6px; border-bottom: 1px solid #E2E8F0; }}
            tr:nth-child(even) {{ background-color: #F8FAFC; }}
            @media print {{
                body {{ padding: 0; }}
                button {{ display: none; }}
            }}
        </style>
    </head>
    <body>
        <div style="display:flex; justify-content:space-between; align-items:center;">
            <h1>AI Stock Research & Thesis Dossier</h1>
            <button onclick="window.print()" style="background:#1E3A8A; color:white; border:none; padding:8px 16px; border-radius:4px; cursor:pointer; font-weight:bold;">🖨️ Print / Save as PDF</button>
        </div>
        <div class="subtitle">Strategy: <b>{strategy_name}</b> | Generated via AI Agent Evaluation Engine</div>
        <table>
            <thead>
                <tr>
                    <th style="text-align:center;">#</th>
                    <th>Company</th>
                    <th style="text-align:right;">CMP</th>
                    <th style="text-align:right;">P/E</th>
                    <th style="text-align:right;">Ind P/E</th>
                    <th style="text-align:right;">PEG</th>
                    <th style="text-align:right;">ROCE %</th>
                    <th style="text-align:right;">5Y ROE %</th>
                    <th style="text-align:center;">Verdict</th>
                </tr>
            </thead>
            <tbody>
                {rows_html}
            </tbody>
        </table>
        <h3 style="color: #0F172A; margin-top: 14px;">Detailed Investment Rationale</h3>
        {details_html}
    </body>
    </html>
    """

    with open(filename, "w", encoding="utf-8") as f:
        f.write(html_content)
    
    return filename

In [8]:
# 1. Load Data
df = load_screener_data("companies.xlsx")

# 2. Setup Dropdown Widget
dropdown = widgets.Dropdown(
    options=[
        "Growth at Reasonable Price (GARP)",
        "High Quality Compounders (Buffett Style)",
        "Deep Value & Cash Cows",
        "Debt-Free / Low-Risk Safe Havens"
    ],
    value="Growth at Reasonable Price (GARP)",
    description="Strategy:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='450px')
)

output_area = widgets.Output()

def on_strategy_change(change):
    with output_area:
        clear_output(wait=True)
        selected_strategy = change['new']
        
        # Evaluate
        results = [evaluate_company(row, selected_strategy) for _, row in df.iterrows()]
        results.sort(key=lambda x: x['Score'], reverse=True)
        
        # Save Report
        report_file = create_printable_report(results, selected_strategy)
        
        # Display Download/Print Link
        display(HTML(f"""
        <div style="background-color: #EFF6FF; border-left: 4px solid #3B82F6; padding: 10px 14px; border-radius: 4px; margin: 10px 0;">
            <b>Report Generated:</b> <a href="{report_file}" target="_blank" style="color: #1D4ED8; font-weight: bold; text-decoration: underline;">Open '{report_file}'</a> (Click <i>'Print / Save as PDF'</i> inside to save as PDF).
        </div>
        """))
        
        # Display Leaderboard
        summary_rows = [{
            "Rank": i + 1,
            "Company": r["Company"],
            "Verdict": r["Verdict"],
            "Score": r["Score"],
            "CMP (Rs.)": f"{r['CMP Rs.']:,.2f}",
            "P/E": r["P/E"],
            "Ind PE": r["Ind PE"],
            "PEG": r["PEG"],
            "ROCE %": f"{r['ROCE %']:.1f}%",
            "5Y ROE %": f"{r['ROE 5Yr %']:.1f}%"
        } for i, r in enumerate(results)]
        
        display(HTML(f"<h3>Top Stock Recommendations ({selected_strategy})</h3>"))
        display(pd.DataFrame(summary_rows).set_index("Rank"))
        
        # Display Detailed Catalysts and Risks
        display(HTML("<h3>Detailed Company Rationale & Risk Assessment</h3>"))
        for rank, item in enumerate(results, start=1):
            pos_bullets = "".join([f"<li style='color:#065F46;'>{c}</li>" for c in item['Catalysts']]) or "<li>None</li>"
            neg_bullets = "".join([f"<li style='color:#991B1B;'>{risk}</li>" for risk in item['Risks']]) or "<li>None</li>"
            
            box_html = f"""
            <div style="border: 1px solid #CBD5E1; border-radius: 6px; padding: 12px; margin-bottom: 12px; background: #FFFFFF;">
                <div style="font-size: 15px; font-weight: bold; color: #1E3A8A;">#{rank} | {item['Company']} — <i>{item['Verdict']}</i> (Score: {item['Score']} pts)</div>
                <div style="font-size: 12px; color: #64748B; margin: 4px 0 8px 0;">CMP: ₹{item['CMP Rs.']:,.2f} | P/E: {item['P/E']:.1f}x | Ind P/E: {item['Ind PE']:.1f}x | ROCE: {item['ROCE %']:.1f}% | 5Y ROE: {item['ROE 5Yr %']:.1f}%</div>
                <table style="width:100%; border:none;">
                    <tr style="background:none;">
                        <td style="width:50%; vertical-align:top; border:none; padding: 0 8px 0 0;">
                            <b style="color: #065F46; font-size: 12px;">Growth Drivers & Moats:</b>
                            <ul style="font-size: 12px; margin: 4px 0 0 16px;">{pos_bullets}</ul>
                        </td>
                        <td style="width:50%; vertical-align:top; border:none; padding: 0 0 0 8px;">
                            <b style="color: #991B1B; font-size: 12px;">Key Headwinds & Risks:</b>
                            <ul style="font-size: 12px; margin: 4px 0 0 16px;">{neg_bullets}</ul>
                        </td>
                    </tr>
                </table>
            </div>
            """
            display(HTML(box_html))

# Connect dropdown callback
dropdown.observe(on_strategy_change, names='value')

# Render UI
display(dropdown)
display(output_area)

# Trigger initial view
on_strategy_change({'new': dropdown.value})

Dropdown(description='Strategy:', layout=Layout(width='450px'), options=('Growth at Reasonable Price (GARP)', …

Output()